In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

True

In [3]:
# Initialize Ollama model
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY"),
    temperature=0.7
)

In [4]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluate: int

In [5]:
def create_outline(state: BlogState) -> BlogState:

    #fetch title
    title = state['title']

    #call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    #update state
    state['outline'] = outline

    return state

In [6]:
def create_blog(state: BlogState) -> BlogState:

    title = state['title']

    outline = state['outline']

    prompt = f'write a detailed blog on the title - {title} using the following outline \n {outline}'

    content = model.invoke(prompt).content

    state['content'] = content

    return state

In [7]:
def evaluate_node(state: BlogState) -> BlogState:

    title = state['title']

    outline = state['outline']

    content = state['content']

    prompt = f"based on outline - {outline} try to rate my blog - {content}"

    evaluate = model.invoke(prompt).content

    state['evaluate'] = evaluate

    return state

In [8]:
graph = StateGraph(BlogState)

#nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate', evaluate_node)

#Edges

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate')
graph.add_edge('evaluate',END)

workflow = graph.compile()


In [9]:
initial_state = {'title': 'Rise an AI in india'}

final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise an AI in india', 'outline': 'Here\'s a detailed outline for a blog post on "The Rise of AI in India," designed to be comprehensive, engaging, and informative.\n\n---\n\n## Blog Title Options:\n\n*   **The AI Awakening: How India is Forging its Future with Artificial Intelligence**\n*   **India\'s AI Ascent: From Niche Technology to National Priority**\n*   **Beyond Silicon Valley: Unpacking the Rise of AI Innovation in India**\n*   **The AI Revolution in India: Opportunities, Challenges, and the Road Ahead**\n\n---\n\n## I. Introduction: Setting the Stage (Approx. 150-200 words)\n\n*   **A. Hook:** Start with a compelling statement about the global AI revolution and its transformative power.\n    *   *Example:* "Artificial Intelligence is no longer a futuristic concept; it\'s a present-day reality reshaping industries, economies, and societies worldwide."\n*   **B. Introduce India\'s Position:** Pivot to India\'s unique and rapidly evolving role in this global AI landsc

In [10]:
print(final_state['outline'])

Here's a detailed outline for a blog post on "The Rise of AI in India," designed to be comprehensive, engaging, and informative.

---

## Blog Title Options:

*   **The AI Awakening: How India is Forging its Future with Artificial Intelligence**
*   **India's AI Ascent: From Niche Technology to National Priority**
*   **Beyond Silicon Valley: Unpacking the Rise of AI Innovation in India**
*   **The AI Revolution in India: Opportunities, Challenges, and the Road Ahead**

---

## I. Introduction: Setting the Stage (Approx. 150-200 words)

*   **A. Hook:** Start with a compelling statement about the global AI revolution and its transformative power.
    *   *Example:* "Artificial Intelligence is no longer a futuristic concept; it's a present-day reality reshaping industries, economies, and societies worldwide."
*   **B. Introduce India's Position:** Pivot to India's unique and rapidly evolving role in this global AI landscape.
    *   Highlight its initial perception vs. current reality.


In [11]:
print(final_state['content'])

## The AI Revolution in India: Opportunities, Challenges, and the Road Ahead

Artificial Intelligence is no longer a futuristic concept; it's a present-day reality reshaping industries, economies, and societies worldwide at an unprecedented pace. From automating complex tasks to powering personalized experiences, AI's transformative power is undeniable. While the global narrative often spotlights tech giants in the West, India has quietly, yet rapidly, emerged as a formidable force in this global AI landscape. Once perceived as primarily an IT services hub, India is now a hotbed of AI innovation, driven by a unique blend of demographic advantages, strategic government initiatives, and a pressing need to solve its own large-scale national challenges.

This blog post will delve into the profound ascent of AI in India, exploring the unique catalysts fueling its growth, the diverse sectors it's revolutionizing, the significant hurdles that lie ahead, and India's distinctive philosophy that

In [12]:
print(final_state['evaluate'])

This is an **outstanding** blog post that adheres exceptionally well to the detailed outline provided. You've done a fantastic job transforming the structural framework into a rich, informative, and engaging piece of content.

Here's a breakdown of the rating based on the outline:

**1. Blog Title:**
*   **Rating: 5/5**
*   You chose "The AI Revolution in India: Opportunities, Challenges, and the Road Ahead," which was one of the excellent options provided in the outline. It perfectly encapsulates the blog's scope.

**2. I. Introduction: Setting the Stage (Approx. 150-200 words)**
*   **Rating: 5/5**
*   **A. Hook:** Excellent, starts with a strong, compelling statement matching the example.
*   **B. Introduce India's Position:** Clearly pivots to India's evolving role and perception.
*   **C. Thesis Statement:** Very clear, comprehensive, and effectively states the blog's main argument and scope.
*   **D. What to Expect:** Naturally covered by the thesis.
*   **Word Count:** Approxima